In [0]:
# Configurações
catalog = "v_credit_medallion"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Schemas completos
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Parâmetro de negócio
JANELA_RECONTATO_DIAS = 7   # recontato: chamado anterior do mesmo cliente e motivo em até 7 dias

# Exportação de CSVs (ligada por padrão, como na versão anterior; use exportar_csv=false para pular)
dbutils.widgets.text("exportar_csv", "true")
EXPORTAR_CSV = dbutils.widgets.get("exportar_csv").strip().lower() == "true"

# Fuso horário da sessão (registrado no manifesto da exportação)
fuso_horario = spark.conf.get("spark.sql.session.timeZone")

In [0]:

from pyspark.sql import functions as F
import plotly.express as px

# Fonte principal do dashboard
GOLD = "v_credit_medallion.gold"
fato = spark.table(f"{GOLD}.fato_chamados")

# Canais que possuem atendimento humano
CANAIS_HUMANOS = [
    "Atendimento Inicial",
    "Atendimento Especializado",
]

# DataFrame geral: uma linha por chamado, conforme a granularidade da Gold
base = fato.select(
    "id_chamado",
    "data_abertura",
    "nome_canal",
    "custo_total",
    "resolvido",
    "abandonado",
    "tempo_espera_segundos",
    "nota_media",
)

# KPIs gerais
geral = base.agg(
    F.count("*").alias("total_chamados"),
    F.sum("custo_total").alias("custo_total"),
    F.avg("custo_total").alias("custo_medio"),
    F.avg(F.col("resolvido").cast("double")).alias("taxa_resolucao"),
    F.avg("nota_media").alias("satisfacao_media"),
    F.count("nota_media").alias("chamados_com_nota"),
).first()

# KPIs específicos de atendimento humano
humanos = base.filter(F.col("nome_canal").isin(CANAIS_HUMANOS))

indicadores_humanos = humanos.agg(
    F.count("*").alias("total_humanos"),
    F.avg(F.col("abandonado").cast("double")).alias("taxa_abandono"),
    F.avg("tempo_espera_segundos").alias("espera_media_segundos"),
    F.percentile_approx(
        "tempo_espera_segundos", 0.5, 10000
    ).alias("espera_mediana_segundos"),
    F.count("tempo_espera_segundos").alias("chamados_com_espera"),
).first()

# Agregação mensal
mensal_spark = (
    base
    .withColumn("mes", F.date_format("data_abertura", "yyyy-MM"))
    .groupBy("mes")
    .agg(
        F.count("*").alias("chamados"),
        F.sum("custo_total").alias("custo_total"),
        F.avg("custo_total").alias("custo_medio"),
    )
    .orderBy("mes")
)

# Plotly trabalha com Pandas; aqui são apenas 19 linhas mensais
mensal = mensal_spark.toPandas()

# Formatação dos meses para exibição
meses_pt = {
    "01": "Jan", "02": "Fev", "03": "Mar",
    "04": "Abr", "05": "Mai", "06": "Jun",
    "07": "Jul", "08": "Ago", "09": "Set",
    "10": "Out", "11": "Nov", "12": "Dez",
}

mensal["mes_label"] = mensal["mes"].apply(
    lambda x: f"{meses_pt[x[5:7]]}/{x[2:4]}"
)

print("Dados preparados.")
print(f"Chamados: {geral['total_chamados']:,}")
print(f"Meses disponíveis: {len(mensal)}")
print(f"Custo total: R$ {geral['custo_total']:,.2f}")


In [0]:

from IPython.display import display, HTML

def moeda(valor):
    return "—" if valor is None else f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def percentual(valor):
    return "—" if valor is None else f"{valor:.2%}".replace(".", ",")

def duracao(segundos):
    if segundos is None:
        return "—"
    total = int(round(segundos))
    minutos, segundos = divmod(total, 60)
    horas, minutos = divmod(minutos, 60)
    if horas:
        return f"{horas}h {minutos:02d}min"
    return f"{minutos}min {segundos:02d}s"

cartoes = [
    ("Chamados", f"{geral['total_chamados']:,}".replace(",", ".")),
    ("Custo total", moeda(geral["custo_total"])),
    ("Custo médio por chamado com custo", moeda(geral["custo_medio"])),
    ("Taxa de resolução registrada", percentual(geral["taxa_resolucao"])),
    ("Abandono nos canais humanos", percentual(indicadores_humanos["taxa_abandono"])),
    ("Espera média humana", duracao(indicadores_humanos["espera_media_segundos"])),
    ("Espera mediana humana", duracao(indicadores_humanos["espera_mediana_segundos"])),
    ("Satisfação média", (
        "—" if geral["satisfacao_media"] is None
        else f"{geral['satisfacao_media']:.2f} / 5".replace(".", ",")
    )),
]

html = """
<div style="
display:grid;
grid-template-columns:repeat(auto-fit,minmax(190px,1fr));
gap:12px;
font-family:Arial,sans-serif;
">
"""

for titulo, valor in cartoes:
    html += f"""
    <div style="
        border:1px solid #d9d9d9;
        border-radius:10px;
        padding:18px;
        min-height:78px;
    ">
        <div style="font-size:13px;color:#666;margin-bottom:12px;">
            {titulo}
        </div>
        <div style="font-size:25px;font-weight:600;">
            {valor}
        </div>
    </div>
    """

html += "</div>"
display(HTML(html))


In [0]:

# Gráfico 1: volume mensal de chamados
fig_volume = px.line(
    mensal,
    x="mes_label",
    y="chamados",
    markers=True,
    title="Volume de chamados por mês",
    labels={
        "mes_label": "Mês",
        "chamados": "Quantidade de chamados",
    },
)

fig_volume.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="Chamados",
    hovermode="x unified",
)

fig_volume.show()


# Gráfico 2: custo total mensal
fig_custo = px.bar(
    mensal,
    x="mes_label",
    y="custo_total",
    title="Custo total por mês",
    labels={
        "mes_label": "Mês",
        "custo_total": "Custo total (R$)",
    },
)

fig_custo.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="Custo (R$)",
    hovermode="x",
)

fig_custo.update_traces(
    hovertemplate="%{x}<br>Custo: R$ %{y:,.2f}<extra></extra>"
)

fig_custo.show()
